# 🎯 Entrenador Interactivo de YOLO
### YOLOv5 · YOLOv8 · YOLOv9 · YOLOv10 · YOLOv11

---
**Antes de empezar:**
1. Andá a `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución`
2. Seleccioná **GPU T4** (gratis) y guardá
3. Ejecutá las celdas de arriba hacia abajo en orden ✅

> Ejecutar una celda: hacé clic en el botón ▶ a la izquierda, o `Shift + Enter`

---
## Celda 1 · Verificar GPU
Verificamos que Colab nos asignó una GPU para entrenar más rápido.

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU disponible:')
    # Mostrar solo la línea con el nombre de la GPU
    for line in result.stdout.split('\n'):
        if 'MiB' in line or 'GeForce' in line or 'Tesla' in line or 'T4' in line or 'A100' in line:
            print(' ', line.strip())
    print()
    print('El entrenamiento será mucho más rápido que en CPU 🚀')
else:
    print('⚠️  No se detectó GPU.')
    print('   Andá a Entorno de ejecución → Cambiar tipo de entorno → GPU T4')


---
## Celda 2 · Instalar librerías
Instalamos todo lo necesario (esto puede tardar 1-2 minutos la primera vez).

In [ ]:
print('📦 Instalando librerías...')
!pip install ultralytics roboflow -q
print()
print('✅ Todo instalado correctamente.')


---
## Celda 3 · ⚙️ CONFIGURACIÓN PRINCIPAL

**Editá los valores entre comillas según tus preferencias antes de ejecutar.**

### Versiones de YOLO disponibles:
| Opción | Versión | Cuándo usarla |
|--------|---------|---------------|
| `"5"` | YOLOv5 | Clásico estable, muy testeado |
| `"8"` | YOLOv8 | Moderno y equilibrado (**RECOMENDADO**) |
| `"9"` | YOLOv9 | Alta precisión, más lento |
| `"10"` | YOLOv10 | Eficiente para producción |
| `"11"` | YOLOv11 | El más nuevo |

### Tamaños de modelo:
| Opción | Tamaño | Para qué sirve |
|--------|--------|----------------|
| `"n"` | Nano | Más rápido, menos preciso |
| `"s"` | Small | Rápido y razonablemente preciso |
| `"m"` | Medium | Equilibrado (**RECOMENDADO**) |
| `"l"` | Large | Más preciso, más lento |
| `"x"` | XLarge | Máxima precisión |

### Épocas (cuánto aprende el modelo):
| Valor | Descripción |
|-------|-------------|
| `50`  | Rápido, resultado básico |
| `100` | Equilibrado (**RECOMENDADO**) |
| `200` | Mejor resultado, más lento |
| `300` | Máximo aprendizaje |

In [ ]:
# ╔══════════════════════════════════════════════╗
# ║         EDITÁ ESTOS VALORES                  ║
# ╚══════════════════════════════════════════════╝

# Versión de YOLO: "5", "8", "9", "10" o "11"
VERSION_YOLO = "8"

# Tamaño del modelo: "n", "s", "m", "l" o "x"
TAMANIO = "m"

# Épocas (cuánto aprende): 50, 100, 200, 300...
# Subido de 10 a 100: con solo 10 épocas las curvas de loss todavía
# estaban bajando fuerte y las métricas (precision/recall/mAP) muy
# ruidosas y bajas — el modelo no había convergido. patience=5 en la
# celda de entrenamiento corta antes si deja de mejorar.
EPOCHS = 100

# Tamaño de imagen en píxeles: 320, 416, 640, 800, 1280
# Subido de 224 a 640: 224px era el tamaño usado en el paper para
# UN recorte de una sola colonia. Acá estás detectando sobre la
# placa completa con muchas colonias chicas — a 224px quedan
# comprimidas a pocos píxeles y son casi imposibles de detectar.
IMGSZ = 640

# ──────────────────────────────────────────────
# Tu API Key de Roboflow
# (roboflow.com → Settings → Roboflow API → Private API Key)
ROBOFLOW_API_KEY = ""  # NO escribir la key acá: usar los Secrets de Colab #TU API KEY AQUI

# Link de tu dataset en Roboflow
# Ejemplo: "https://app.roboflow.com/mi-workspace/mi-proyecto/1"
ROBOFLOW_LINK = "https://app.roboflow.com/ucuestudiantesproyecto-gmail-com/subdataset_v2-ucu-udelar-2026-a/5" # ejemplo: "https://app.roboflow.com/WORKSPACE/PROYECTO/VERSION"

# ══════════════════════════════════════════════
#   NO MODIFICAR DESDE AQUÍ HACIA ABAJO
# ══════════════════════════════════════════════

nombres_tamanio = {"n":"Nano","s":"Small","m":"Medium","l":"Large","x":"XLarge"}

print('=' * 50)
print('  RESUMEN DE CONFIGURACIÓN')
print('=' * 50)
print(f'  Versión YOLO     : YOLOv{VERSION_YOLO}')
print(f'  Tamaño de modelo : {nombres_tamanio.get(TAMANIO, TAMANIO)} ({TAMANIO})')
print(f'  Épocas           : {EPOCHS}')
print(f'  Tamaño de imagen : {IMGSZ}px')
print(f'  API Key          : {ROBOFLOW_API_KEY[:6]}...(oculto)')
print(f'  Dataset          : {ROBOFLOW_LINK}')
print('=' * 50)

# Validaciones básicas
errores = []
if VERSION_YOLO not in ["5","8","9","10","11"]:
    errores.append('VERSION_YOLO debe ser "5", "8", "9", "10" o "11"')
if TAMANIO not in ["n","s","m","l","x"]:
    errores.append('TAMANIO debe ser "n", "s", "m", "l" o "x"')
if "roboflow.com" not in ROBOFLOW_LINK:
    errores.append('ROBOFLOW_LINK no parece ser un link de Roboflow válido')
if ROBOFLOW_API_KEY == "TU_API_KEY_AQUI":
    errores.append('Tenés que reemplazar ROBOFLOW_API_KEY con tu clave real')

if errores:
    print()
    print('❌ HAY ERRORES EN LA CONFIGURACIÓN:')
    for e in errores:
        print(f'   • {e}')
    print()
    print('Corregí los valores y volvé a ejecutar esta celda.')
else:
    print()
    print('✅ Configuración válida. Podés pasar a la siguiente celda.')


---
## Celda 4 · Descargar el dataset de Roboflow
Descargamos tu dataset usando las credenciales configuradas arriba.

In [ ]:
from roboflow import Roboflow
import os

# Parsear el link de Roboflow
partes = ROBOFLOW_LINK.rstrip('/').split('/')
try:
    version_num = int(partes[-1])
    proyecto    = partes[-2]
    workspace   = partes[-3]
except (ValueError, IndexError):
    raise ValueError(
        'No se pudo parsear el link de Roboflow.\n'
        'Asegurate de que termine con el número de versión.\n'
        'Ejemplo: https://app.roboflow.com/mi-workspace/mi-proyecto/1'
    )

print(f'Workspace : {workspace}')
print(f'Proyecto  : {proyecto}')
print(f'Versión   : {version_num}')
print()
print('Descargando dataset...')

rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(workspace).project(proyecto)
dataset = project.version(version_num).download('yolov8')

DATASET_DIR = dataset.location
DATA_YAML   = os.path.join(DATASET_DIR, 'data.yaml')

# Buscar data.yaml si no está en la raíz
if not os.path.exists(DATA_YAML):
    for root, _, files in os.walk(DATASET_DIR):
        if 'data.yaml' in files:
            DATA_YAML = os.path.join(root, 'data.yaml')
            break

print()
print(f'✅ Dataset descargado en: {DATASET_DIR}')
print(f'   Archivo de datos: {DATA_YAML}')


# Guardar el proceso en Google Drive

**Montar Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Carpeta donde se guardará todo
DRIVE_DIR = '/content/drive/MyDrive/yolo_entrenamiento'

import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'✅ Drive montado. Resultados se guardarán en: {DRIVE_DIR}')

---
## Celda 5 · 🚀 ¡Entrenar el modelo!
Acá empieza el entrenamiento. Puede tardar desde minutos hasta horas dependiendo de:
- El tamaño de tu dataset
- El tamaño del modelo elegido
- Las épocas configuradas

> No cierres la pestaña de Colab mientras entrena.

In [ ]:
from ultralytics import YOLO

nombre_modelo_map = {
    '5':  f'yolov5{TAMANIO}.pt',
    '8':  f'yolov8{TAMANIO}.pt',
    '9':  f'yolov9{TAMANIO}.pt',
    '10': f'yolov10{TAMANIO}.pt',
    '11': f'yolo11{TAMANIO}.pt',
}
nombre_modelo = nombre_modelo_map[VERSION_YOLO]

print(f'Cargando modelo base: {nombre_modelo}')
print(f'Entrenando YOLOv{VERSION_YOLO}-{TAMANIO.upper()} por {EPOCHS} épocas...')
print()

model = YOLO(nombre_modelo)

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    project=DRIVE_DIR,
    #project='runs/train',
    name=f'yolov{VERSION_YOLO}{TAMANIO}_colab',
    exist_ok=True,
    device=0,  # Usa la GPU de Colab
    save_period=10,
    batch=32,

    # --- Optimizador (según paper) ---
    optimizer='Adam',
    lr0=0.0001,          # paper: 1e-4
    weight_decay=0.0001, # paper: L2 = 1e-4
    # momentum no se toca: 0.937 es el default de Ultralytics
    # (se usa como beta1 de Adam); el paper no especifica un valor.

    # --- Augmentación (según paper, on-the-fly) ---
    degrees=12.0,   # paper: rotación ±12°
    translate=0.1,  # paper: traslación ≤10% (ya es el default de Ultralytics)
    scale=0.1,      # paper: escala isotrópica 0.90–1.10× (default de Ultralytics es 0.5, mucho más agresivo)
    hsv_v=0.15,     # paper: brillo/contraste 0.85–1.15 (default de Ultralytics es 0.4)
    hsv_h=0.0,      # el paper no varía el matiz; el color es un rasgo diagnóstico de especie, mejor no tocarlo
    hsv_s=0.0,      # ídem para saturación
    fliplr=0.5,     # paper: flip horizontal p=0.5 (ya es el default de Ultralytics)
    # flipud=0.5,   # opcional (no viene del paper): una placa fotografiada desde arriba
                     # no tiene una orientación fija, así que un flip vertical también
                     # es una augmentación válida para este caso. Descomentar si querés probarlo.

    close_mosaic=10,  # se desactiva el mosaico en las últimas 10 épocas (antes era 20 con solo 10
                       # épocas totales, por lo que nunca llegaba a activarse)
    patience=15,       # paper usa 5, pero las métricas de detección son más ruidosas época a
                       # época que las de un clasificador — 15 da más margen antes de cortar.
                       # Si preferís fidelidad estricta al paper, volvé a patience=5.

    # NOTA: conf e iou se sacaron de acá. Ultralytics los reutiliza para la
    # validación interna de cada época — forzar conf=0.4 ahí filtraba
    # detecciones de confianza media/baja también DURANTE el entrenamiento,
    # perjudicando el recall que ves en las gráficas. Usalos en model.predict()
    # cuando pruebes el modelo ya entrenado, no acá.
)

SAVE_DIR = str(results.save_dir)
print()
print('=' * 50)
print('  ✅ ¡ENTRENAMIENTO COMPLETADO!')
print('=' * 50)
print(f'  Resultados guardados en: {SAVE_DIR}')
print(f'  Mejor modelo: {SAVE_DIR}/weights/best.pt')


In [ ]:
# En caso de que se cierre el entorno de ejecución
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Celda 6 en caso de que se cierre el entorno de ejecucion
from IPython.display import Image, display
import os

# Definí manualmente SAVE_DIR apuntando a Drive
SAVE_DIR = '/content/drive/MyDrive/yolo_entrenamiento/yolov8m_colab'

results_png = os.path.join(SAVE_DIR, 'results.png')
if os.path.exists(results_png):
    print('📊 Gráfico de entrenamiento:')
    display(Image(filename=results_png, width=900))

val_img = os.path.join(SAVE_DIR, 'val_batch0_pred.jpg')
if os.path.exists(val_img):
    print()
    print('🖼️  Ejemplos de detecciones en validación:')
    display(Image(filename=val_img, width=700))

---
## Celda 6 · Ver resultados del entrenamiento
Mostramos las métricas y gráficos generados.

In [ ]:
from IPython.display import Image, display
import os

# Mostrar gráfico de entrenamiento
results_png = os.path.join(SAVE_DIR, 'results.png')
if os.path.exists(results_png):
    print('📊 Gráfico de entrenamiento:')
    display(Image(filename=results_png, width=900))
else:
    print('No se encontró results.png todavía.')

# Mostrar ejemplos de predicciones de validación
val_img = os.path.join(SAVE_DIR, 'val_batch0_pred.jpg')
if os.path.exists(val_img):
    print()
    print('🖼️  Ejemplos de detecciones en validación:')
    display(Image(filename=val_img, width=700))


---
## Celda 7 · Descargar el modelo entrenado
Descargá el archivo `best.pt` a tu computadora para usarlo después.

In [ ]:
from google.colab import files
import os

best_pt = os.path.join(SAVE_DIR, 'weights', 'best.pt')

if os.path.exists(best_pt):
    print(f'Descargando: {best_pt}')
    files.download(best_pt)
    print('✅ Descarga iniciada. Revisá la carpeta de descargas de tu navegador.')
else:
    print('❌ No se encontró best.pt. ¿El entrenamiento terminó correctamente?')
    print(f'   Buscando en: {best_pt}')


---
## Celda 8 (OPCIONAL) · Probar el modelo con una imagen
Podés probar tu modelo con cualquier imagen antes de descargarlo.

In [ ]:
from google.colab import files as colab_files
from ultralytics import YOLO
from IPython.display import Image as IPImage, display
import os

print('Subí una imagen para probar el modelo...')
uploaded = colab_files.upload()

if uploaded:
    imagen_path = list(uploaded.keys())[0]
    print(f'Imagen subida: {imagen_path}')
    print('Ejecutando detección...')

    model_test = YOLO(os.path.join(SAVE_DIR, 'weights', 'best.pt'))
    resultados = model_test.predict(imagen_path, conf=0.5, iou=0.45, save=True, project='predicciones', name='test')

    # Mostrar resultado
    for r in resultados:
        if r.save_dir:
            pred_path = os.path.join(r.save_dir, os.path.basename(imagen_path))
            if os.path.exists(pred_path):
                print()
                print(f'Objetos detectados: {len(r.boxes)}')
                display(IPImage(filename=pred_path, width=700))
else:
    print('No subiste ninguna imagen.')


---
## ¿Qué hago ahora con el modelo descargado?

Con el archivo `best.pt` podés:

```python
from ultralytics import YOLO

# Cargar el modelo entrenado
model = YOLO('best.pt')

# Detectar objetos en una imagen
results = model('mi_foto.jpg')
results[0].show()  # Ver los resultados
```

O para detección en tiempo real con la cámara:
```python
model.predict(source=0, show=True)  # 0 = cámara web
```

# Adicional - Gráficos
Gráficos adicionales sobre los resultados del modelo

In [ ]:
from google.colab import files
import os

# Lista de archivos adicionales para descargar
additional_files = [
    'confusion_matrix.png',
    'F1_curve.png',
    'P_curve.png',
    'R_curve.png',
    'PR_curve.png',
    'labels.jpg'
]

print('Descargando archivos adicionales:')
for filename in additional_files:
    filepath = os.path.join(SAVE_DIR, filename)
    if os.path.exists(filepath):
        print(f'Descargando: {filename}')
        files.download(filepath)
        print(f'OK: Descarga de {filename} iniciada.')
    else:
        print(f'ERROR: No se encontró {filename}.')
print('\nTodas las descargas han sido iniciadas. Revisa la carpeta de descargas de tu navegador.')